# Lecture 2: Claim Cards and the R0-R5 Risk Model

A claim card is a machine-readable assurance artifact. It records what was checked, what theorem names were involved, what axioms were observed, what exclusions remain, what trusted base is assumed, and what risk score follows.

A claim card is not a marketing page. It is a structured input to policy.


## Learning Objectives

- Read the claim card schema.
- Explain risk levels R0 through R5.
- Generate an offline fixture claim card.
- Understand why R3 can authorize lower-layer library use but not wallet construction.
- Identify blockers and deployment constraints in a claim card.


## Risk Levels

- `R0`: Unknown or untrusted. No usable evidence.
- `R1`: Tests, audits, or informal claims only.
- `R2`: Formal model exists, but incomplete, weakly tied to production code, or major proof gaps remain.
- `R3`: A specific lower-layer implementation artifact is Lean-checked for a specific backend and theorem boundary.
- `R4`: End-to-end primitive proof covers public API, parsing/encoding, scalar arithmetic, hashing interface, signature equation, rejection rules, and implementation boundary.
- `R5`: R4 plus reproducible production builds, compiler/build assurance, side-channel analysis, hardware/KMS/MPC integration, and operational controls.

Ed25519 field plus Edwards arithmetic alone reaches R3. Since the corpus completed its four-tier signature apex (2026-07-06), the FULL configured certificate set - arithmetic, scalars, encoding/decoding, and the four apex tiers, each with its axiom cone pinned to the fork's documented boundary - reaches **R4**, always with explicit residual blockers (the SHA-512 oracle, hypothesis-parametric wire parses, translation faithfulness, and the missing side-channel/build assurance that gates R5).


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.claims import build_claim_card
from pacta.config import load_config

config = load_config(repo_root / "examples" / "repos.yaml")
repo = config.repo_named("dalek-ed25519-verified")
card = build_claim_card(repo, repo_root / "repos" / repo.name, offline_fixture=True)

print(card["component"])
print(card["risk"]["level"])
print(card["risk"]["rationale"])
print(card["certificates"][0])


In [ ]:
important_fields = [
    "component",
    "repo_url",
    "repo_commit",
    "verification_dir",
    "kind",
    "verified_backend",
    "certificates",
    "guarantees",
    "preconditions",
    "exclusions",
    "trusted_base",
    "evidence",
    "risk",
]
for field in important_fields:
    print(field, "=>", type(card.get(field)).__name__)


## Reading a Certificate Entry

A certificate entry contains:

- `name`: theorem or aggregate certificate name.
- `status`: `proven`, `missing`, `failed`, or `unknown`.
- `axiom_status`: `clean`, `dirty`, or `not_checked`.
- `observed_axioms`: axioms reported by Lean.
- `expected_axioms`: allowed standard axioms for this profile.

A clean result requires more than a theorem name. It requires a successful replay or trusted attestation, the RIGHT axiom set per certificate (standard-three below the apex, the fork's documented boundary at the apex tiers - deviation in either direction is dirty), and no policy-blocking exclusions.

### The ratchet, run both ways

First the napkin: a two-certificate card you can score in your head. Then the real thing: the shipped R4 fixture with sixteen certificates and per-tier boundary axioms.


In [ ]:
# NAPKIN: an arithmetic-only card. Two certificates, standard axioms.
from pacta.risk import score_claim_card

napkin_card = {
    "kind": "ed25519",
    "certificates": [
        {"name": "CurveFieldProofs.fieldImplementation", "status": "proven", "axiom_status": "clean"},
        {"name": "CurveFieldProofs.edwardsImplementation", "status": "proven", "axiom_status": "clean"},
    ],
    "exclusions": ["full EdDSA signature verification"],
    "meta": {"r4_requirements": []},
}
napkin = score_claim_card(napkin_card)
print(napkin.level, "-", napkin.rationale)


In [ ]:
# REAL: the shipped R4 fixture - sixteen certificates, apex tiers carrying
# the dalek fork's documented boundary axioms. Same scoring function.
from pacta.yamlio import load_data

real_card = load_data(repo_root / "examples" / "dalek-ed25519.claims.yaml")
real = score_claim_card(real_card)
print(real.level)
print(real.rationale[:180], "...")
print("residual blockers:")
for blocker in real.blockers:
    print(" -", blocker)
apex = [c for c in real_card["certificates"] if c["name"].endswith("_decompress")][0]
print("full-lift tier expects:", apex["expected_axioms"])


In [ ]:
for cert in card["certificates"]:
    print(f"{cert['name']}: {cert['status']} / {cert['axiom_status']}")
    print("  observed:", cert["observed_axioms"])
    print("  expected:", cert["expected_axioms"])


## Deployment Constraints

Deployment constraints are where many assurance cases become honest. For Ed25519 arithmetic, constraints include:

- Use exact pinned source or reviewed diff.
- Use verified serial/u64 backend only.
- Disable accelerator/syscall/hardware/SIMD paths unless separately certified.
- Do not treat this as full EdDSA verification.
- Keep key custody behind HSM/MPC/policy firewall until signing stack proof coverage improves.
- Use ordinary tests/fuzzing at encoding/API/transaction boundaries.


In [ ]:
for constraint in card["risk"]["deployment_constraints"]:
    print("-", constraint)


## Exercises

- Change the generated card in memory so one certificate is `missing`. Rescore it and explain the change.
- Write a short policy that allows `build-library` at R3 but denies `build-wallet-demo` below R4.
- Compare the trusted base for local replay versus third-party attestation.


In [ ]:
from copy import deepcopy
from pacta.risk import score_claim_card

weaker = deepcopy(card)
weaker["certificates"][0]["status"] = "missing"
weaker["certificates"][0]["axiom_status"] = "not_checked"
assessment = score_claim_card(weaker)
print(assessment.level)
print(assessment.rationale)
print(assessment.blockers)
